# Step 22 — Does mass error explain the location error too? (CPU re-analysis of saved runs)

**Inputs.** Mount the committed outputs of **Step 19** (`step19_logits.npz`), **Step 21** (`step21_*.npz`) and **Step 20** (`step20_logits.npz`). CPU only, about 2 hours.

**A. Mass propagates to place.** The cusp fold condition $4a^3=27bh^2$ implies that, with $a$ roughly linear in $\log\sigma$ near the transition, $\Delta\log\sigma_{\rm bif}\approx-3(b/4)^{1/3}\,\Delta|h|^{2/3}/s$, where $s=-\mathrm da/\mathrm d\log\sigma$. We test four things:
1. the raw correlation of the location error with $\Delta|h|^{2/3}$;
2. the parameter-free cusp prediction of each pair's location error, using $b$ and $s$ from the data;
3. whether rebalancing class mass collapses the location scatter to the data-versus-data level;
4. a split-half version (asymmetry from one half of the trajectories, location from the other), to rule out shared estimation noise.

All with class-block bootstrap intervals.

**B. The headline, corrected.** Rebalanced-geometry agreement against the Spearman–Brown reliability ceiling, with class-block intervals.

**C. The identity, shown on data alone.** Reweight the *reference* by known factors: $\Delta h$ against $\tfrac12\Delta\log$ should have slope about 1 with no model involved.

**D. Statistics.** Census $h$ per pair with bootstrap p-values and Benjamini–Hochberg FDR; class-block intervals for the hierarchy; the pretrained per-pair table recomputed with bootstrap medians, so every point lies inside its interval.

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt, time
from itertools import combinations
from scipy.signal import fftconvolve
from scipy.stats import spearmanr
plt.rcParams['figure.dpi'] = 120; rng = np.random.default_rng(22)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
def find(pat): p = glob.glob(f'/kaggle/input/**/{pat}', recursive=True); assert p, f'mount the output containing {pat}'; return p[0]
Z19 = np.load(find('step19_logits.npz')); L_TE, W_TE, L_TR, W_TR, L_M, W_M, SIG = [Z19[k] for k in ['L_TE', 'W_TE', 'L_TR', 'W_TR', 'L_M', 'W_M', 'SIG_REC']]
RUNS = {'step19': (L_M, W_M)}
for k in ['baseline', 'mass_correct', 'imposed', 'mass_correct_window']:
    z = np.load(find(f'step21_{k}.npz')); RUNS[k] = (z['L'], z['W'])
print({k: v[1].shape[0] for k, v in RUNS.items()})
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3, c1 + 3*c3*s**2 + 2*c2*s, b
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG))] for a, b in PAIRS}
def analyse(L, W, a, b, idx, balance=False, rw=None, g0=0, slope=False):
    """Along the pair log-odds (reference standardisation): returns dict(sigma, gi, h, a, b[, s]). rw: extra class reweighting factors (for part C)."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if rw is not None: Wa = Wa*rw[a]; Wb = Wb*rw[b]
    if balance: tot = 0.5*(Wa.sum()+Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    def dist(g):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: return None
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; return wkde_score((z - mid)/half, Wa + Wb)
    prev = None if g0 == 0 else 1
    for g in range(g0, len(SIG)):
        d = dist(g); nm = 1 if d is None else n_max(stationary(*d))
        if nm >= 2 and prev is not None:
            h, a_, b_ = cusp(*d); out = dict(sigma=float(SIG[g]), gi=g, h=h, a=a_, b=b_)
            if slope:                                            # a(log sigma) near the transition, from the levels just above and below
                gs = [k for k in range(max(0, g-4), min(len(SIG), g+3)) if dist(k) is not None]
                av = [cusp(*dist(k))[1] for k in gs]
                if len(gs) >= 3: out['s'] = -np.polyfit(np.log(SIG[gs]), av, 1)[0]
            return out
        if nm == 1: prev = 1
    return None
def boot(L, W, a, b, n, B, balance=False, ref=False, g_hint=None):
    """Bootstrap medians and 95% intervals of sigma_bif and h at sample size n."""
    ss, hs = [], []; g0 = 0 if g_hint is None else max(0, g_hint - 8)
    for _ in range(B):
        idx = rng.choice(len(W), size=n, replace=True); r = analyse(L, W, a, b, idx, balance, g0=g0)
        if r: ss.append(r['sigma']); hs.append(r['h'])
    if len(ss) < B//3: return None
    return dict(sigma=np.median(ss), s_lo=np.percentile(ss, 2.5), s_hi=np.percentile(ss, 97.5), h=np.median(hs), h_lo=np.percentile(hs, 2.5), h_hi=np.percentile(hs, 97.5), n_ok=len(ss))
def class_block_ci(x, y, pairs, B=5000, fn=None):
    """Class-block bootstrap: resample the 10 classes with replacement, weight each pair by the product of its classes' multiplicities."""
    P = np.array(pairs); out = []
    for _ in range(B):
        cls = rng.choice(10, 10, replace=True); w = np.array([np.sum(cls == a)*np.sum(cls == b) for a, b in P], float)
        if w.sum() < 6: continue
        if fn is None:
            mx, my = np.average(x, weights=w), np.average(y, weights=w); out.append(np.sum(w*(x-mx)*(y-my))/np.sqrt(np.sum(w*(x-mx)**2)*np.sum(w*(y-my)**2)))
        else: out.append(fn(w))
    out = np.array(out, float); out = out[np.isfinite(out)]
    return np.percentile(out, [2.5, 97.5]) if len(out) > 20 else np.array([np.nan, np.nan])
print('ready')

## A. Mass propagates to place

In [ ]:
def mass_to_place(run, B=100):
    L, W = RUNS[run]; n = len(W); t0 = time.time(); R = {}
    for a, b in PAIRS:
        irf = rng.choice(len(W_TE), size=n, replace=False); d0 = analyse(L_TE, W_TE, a, b, irf, slope=True)
        m0 = analyse(L, W, a, b, np.arange(n)); m0b = analyse(L, W, a, b, np.arange(n), balance=True)
        if not (d0 and m0 and m0b and 's' in d0): continue
        d = boot(L_TE, W_TE, a, b, n, B, g_hint=d0['gi']); m = boot(L, W, a, b, n, B, g_hint=m0['gi']); mb = boot(L, W, a, b, n, B, balance=True, g_hint=m0b['gi'])
        if d and m and mb: R[(a, b)] = dict(d=d, m=m, mb=mb, b=d0['b'], s=d0['s'])
    print(f"{run}: {len(R)} pairs ({(time.time()-t0)/60:.0f} min)"); return R
RES = {k: mass_to_place(k) for k in ['step19', 'baseline']}
def summarize(run):
    R = RES[run]; P = list(R); lr = np.array([np.log(R[p]['m']['sigma']/R[p]['d']['sigma']) for p in P]); lrb = np.array([np.log(R[p]['mb']['sigma']/R[p]['d']['sigma']) for p in P])
    dh23 = np.array([abs(R[p]['m']['h'])**(2/3) - abs(R[p]['d']['h'])**(2/3) for p in P])
    pred = np.array([-3*(R[p]['b']/4)**(1/3)*dh/max(R[p]['s'], 1e-3) for p, dh in zip(P, dh23)])
    ci = class_block_ci(lr, dh23, P)
    print(f"\n[{run}] location error log(sigma_model/sigma_data): raw SD {lr.std():.3f}, median ratio {np.exp(np.median(lr)):.2f}")
    print(f"   corr with Delta|h|^(2/3): {np.corrcoef(lr, dh23)[0,1]:.3f} (Spearman {spearmanr(lr, dh23).correlation:.3f}; class-block 95% CI [{ci[0]:.2f}, {ci[1]:.2f}])")
    ok = np.isfinite(pred) & (np.abs(pred) < 3)
    print(f"   cusp prediction (no free parameters): corr {np.corrcoef(pred[ok], lr[ok])[0,1]:.3f}, slope of measured on predicted {np.polyfit(pred[ok], lr[ok], 1)[0]:.2f} (ideal 1), n={ok.sum()}")
    print(f"   after rebalancing class mass: SD {lrb.std():.3f} (from {lr.std():.3f}), median ratio {np.exp(np.median(lrb)):.2f}; corr with Delta|h|^(2/3) now {np.corrcoef(lrb, dh23)[0,1]:.3f}")
    return P, lr, lrb, dh23, pred
S19 = summarize('step19'); SB = summarize('baseline')
# data-vs-data reference scatter (train vs test reference at the same n)
n = len(RUNS['step19'][1]); dd = []
for a, b in PAIRS:
    r1 = boot(L_TE, W_TE, a, b, min(n, len(W_TE)), 40); r2 = boot(L_TR, W_TR, a, b, min(n, len(W_TR)), 40)
    if r1 and r2: dd.append(np.log(r2['sigma']/r1['sigma']))
print(f"\ndata-vs-data (train vs test reference) SD of log ratio: {np.std(dd):.3f} over {len(dd)} pairs  <- the floor rebalancing should approach")
# split-half: asymmetry from half A, location from half B (independent trajectories and reference halves)
L, W = RUNS['step19']; nh = len(W)//2; perm = rng.permutation(len(W)); A_, B_ = perm[:nh], perm[nh:2*nh]; pr = rng.permutation(len(W_TE)); RA, RB = pr[:nh], pr[nh:2*nh]
x, y, Pp = [], [], []
for a, b in PAIRS:
    mA = analyse(L, W, a, b, A_); dA = analyse(L_TE, W_TE, a, b, RA); mB = analyse(L, W, a, b, B_); dB = analyse(L_TE, W_TE, a, b, RB)
    if mA and dA and mB and dB: x.append(abs(mA['h'])**(2/3) - abs(dA['h'])**(2/3)); y.append(np.log(mB['sigma']/dB['sigma'])); Pp.append((a, b))
ci = class_block_ci(np.array(y), np.array(x), Pp)
print(f"split-half (h from half A, location from half B): corr {np.corrcoef(x, y)[0,1]:.3f}, class-block 95% CI [{ci[0]:.2f}, {ci[1]:.2f}], n={len(x)}")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8)); P, lr, lrb, dh23, pred = S19
ax[0].scatter(dh23, lr, s=14); ax[0].axhline(0, color='k', lw=0.5); ax[0].axvline(0, color='k', lw=0.5); ax[0].set_xlabel('|h_model|^(2/3) - |h_data|^(2/3)'); ax[0].set_ylabel('log(sigma_model / sigma_data)'); ax[0].set_title('location error follows asymmetry error')
ok = np.isfinite(pred) & (np.abs(pred) < 3); ax[1].scatter(pred[ok], lr[ok], s=14); ax[1].axline((0, 0), slope=1, color='k', lw=0.6); ax[1].set_xlabel('cusp prediction'); ax[1].set_ylabel('measured'); ax[1].set_title('parameter-free prediction')
bins = np.linspace(-1.2, 1.2, 25); ax[2].hist(lr, bins, alpha=0.6, label='raw'); ax[2].hist(lrb, bins, alpha=0.6, label='class mass rebalanced'); ax[2].set_xlabel('log(sigma_model / sigma_data)'); ax[2].legend(fontsize=7); ax[2].set_title('rebalancing collapses location error?')
plt.tight_layout(); plt.show()

## B. Rebalanced geometry against the proper ceiling, and C. the identity on data alone

In [ ]:
sb = lambda r: 2*r/(1+r)
for run in ['step19', 'baseline']:
    L, W = RUNS[run]; n = len(W); nh = n//2; perm = rng.permutation(n); pr = rng.permutation(len(W_TE))
    hm1, hm2, hd1, hd2, hmb, hd, P = [], [], [], [], [], [], []
    for a, b in PAIRS:
        r = [analyse(L, W, a, b, perm[:nh], balance=True), analyse(L, W, a, b, perm[nh:2*nh], balance=True), analyse(L_TE, W_TE, a, b, pr[:nh]), analyse(L_TE, W_TE, a, b, pr[nh:2*nh]),
             analyse(L, W, a, b, np.arange(n), balance=True), analyse(L_TE, W_TE, a, b, pr[:n])]
        if all(r): hm1.append(r[0]['h']); hm2.append(r[1]['h']); hd1.append(r[2]['h']); hd2.append(r[3]['h']); hmb.append(r[4]['h']); hd.append(r[5]['h']); P.append((a, b))
    rm, rd = np.corrcoef(hm1, hm2)[0, 1], np.corrcoef(hd1, hd2)[0, 1]; ceil = np.sqrt(sb(rm)*sb(rd)); r = np.corrcoef(hmb, hd)[0, 1]; ci = class_block_ci(np.array(hmb), np.array(hd), P)
    print(f"[{run}] rebalanced h model vs data: r = {r:.3f} (class-block 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]); split-half reliabilities {rm:.3f} / {rd:.3f}; Spearman-Brown ceiling {ceil:.3f}; disattenuated {r/ceil:.3f}")
# C: the Delta h = 1/2 Delta log(w_a/w_b) law on the DATA alone (no model): reweight the reference by known factors
rw = np.exp(rng.uniform(-0.7, 0.7, 10)); n = min(6000, len(W_TE)); idx = rng.choice(len(W_TE), size=n, replace=False); dh, dl = [], []
for a, b in PAIRS:
    r0 = analyse(L_TE, W_TE, a, b, idx); r1 = analyse(L_TE, W_TE, a, b, idx, rw=rw)
    if r0 and r1: dh.append(r1['h'] - r0['h']); dl.append(0.5*np.log(rw[a]/rw[b]))
print(f"identity on the reference alone (known reweighting, no model): Delta h vs half Delta log ratio: slope {np.polyfit(dl, dh, 1)[0]:.2f}, corr {np.corrcoef(dl, dh)[0,1]:.2f}, n={len(dh)}")
print("-> slope ~1 is a property of the coordinate and the fit, i.e. a pipeline check, as the reviewer argued.")

## D. Census statistics (Step 20 logits) and recomputed pretrained per-pair table

In [ ]:
Z20 = np.load(find('step20_logits.npz')); L_MAIN, SIG20, YTE = Z20['L_MAIN'], Z20['SIG'], Z20['YTE']; YT4 = np.tile(YTE, 4)
def kde_score(m, bwf=1.5):
    m = np.clip(m, -8, 8); bw = max(bwf*1.06*m.std()*len(m)**(-1/5), 1e-3); h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def census_z(Z, ya, g0=0):
    seen = g0 > 0
    for g in range(g0, len(SIG20)):
        z = Z[g]; mua, mub = z[ya].mean(), z[~ya].mean(); half = 0.5*(mua-mub); sep = abs(half)/(0.5*(z[ya].std()+z[~ya].std()))
        if sep < 0.3: seen = True; continue
        sm, p = kde_score((z-0.5*(mua+mub))/half); nm = n_max(stationary(sm, p))
        if nm >= 2 and seen: return float(SIG20[g]), cusp(sm, p)[0], g
        if nm == 1: seen = True
    return None
NULLMAX = 0.0468; pv, H, S, Pc = [], [], [], []; t0 = time.time()
for a, b in PAIRS:
    rows = np.where((YT4 == a) | (YT4 == b))[0]; ya = YT4[rows] == a; Z = [L_MAIN[g, rows, a].astype(np.float64) - L_MAIN[g, rows, b] for g in range(len(SIG20))]; r0 = census_z(Z, ya)
    if r0 is None: continue
    ia, ib = np.where(ya)[0], np.where(~ya)[0]; hs, ss = [], []
    for _ in range(500):
        sel = np.concatenate([rng.choice(ia, len(ia)), rng.choice(ib, len(ib))]); r = census_z([z[sel] for z in Z], ya[sel], g0=max(0, r0[2]-8))
        if r: ss.append(r[0]); hs.append(r[1])
    hs = np.array(hs); med = np.median(hs)
    p = 2*min(np.mean(np.abs(hs) <= NULLMAX) if abs(med) > NULLMAX else 1.0, 0.5)          # bootstrap p for |h| exceeding the through-classifier null
    pv.append(max(p, 1/len(hs))); H.append(med); S.append(np.median(ss)); Pc.append((a, b))
print(f"census bootstrap ({(time.time()-t0)/60:.0f} min)")
pv = np.array(pv); order = np.argsort(pv); m = len(pv); thr = 0.05*np.arange(1, m+1)/m; passed = np.zeros(m, bool)
k = np.where(pv[order] <= thr)[0]
if len(k): passed[order[:k.max()+1]] = True
print(f"pairs whose |h| exceeds the through-classifier null: {np.sum(pv < 0.05)}/{m} uncorrected, {passed.sum()}/{m} after Benjamini-Hochberg (FDR 5%)")
S = np.array(S); kd = lambda p: 'va' if ((p[0] in VEH) != (p[1] in VEH)) else ('vv' if p[0] in VEH else 'aa'); K = np.array([kd(p) for p in Pc])
def hier(w): va = K == 'va'; aa = K == 'aa'; return np.exp(np.average(np.log(S[va]), weights=w[va]) - np.average(np.log(S[aa]), weights=w[aa])) if w[va].sum() > 0 and w[aa].sum() > 0 else np.nan
ci = class_block_ci(None, None, Pc, fn=hier); print(f"hierarchy (geometric-mean ratio vehicle/animal over animal/animal) {hier(np.ones(len(Pc))):.2f}, class-block 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]")
# pretrained per-pair table with bootstrap medians (points always inside their intervals)
L, W = RUNS['step19']; n = len(W); print(f"\n{'pair':>20} {'sigma data':>18} {'sigma model':>18} {'ratio (median of paired boot) [95%]':>36} {'h data':>8} {'h model':>8}")
for a, b in PAIRS:
    irf = rng.choice(len(W_TE), size=n, replace=False); d0 = analyse(L_TE, W_TE, a, b, irf); m0 = analyse(L, W, a, b, np.arange(n))
    if not (d0 and m0): continue
    rat, sd_, sm_, hd_, hm_ = [], [], [], [], []
    for _ in range(100):
        r1 = analyse(L, W, a, b, rng.integers(n, size=n), g0=max(0, m0['gi']-8)); r2 = analyse(L_TE, W_TE, a, b, rng.choice(len(W_TE), size=n), g0=max(0, d0['gi']-8))
        if r1 and r2: rat.append(r1['sigma']/r2['sigma']); sd_.append(r2['sigma']); sm_.append(r1['sigma']); hd_.append(r2['h']); hm_.append(r1['h'])
    if len(rat) < 30: continue
    print(f"{CLASSES[a]+'/'+CLASSES[b]:>20} {np.median(sd_):18.3f} {np.median(sm_):18.3f} {np.median(rat):12.2f} [{np.percentile(rat,2.5):.2f}, {np.percentile(rat,97.5):.2f}] {np.median(hd_):+8.3f} {np.median(hm_):+8.3f}")

## What to report back
- **Part A:** the two summary blocks, the data-versus-data line, the split-half line, and the figure.
- **Part B:** the two ceiling lines.
- **Part C:** the identity line.
- **Part D:** the FDR line, the hierarchy line and the per-pair table.